# Audit logging with the Elastic Stack: retain and investigate

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/audit-logging-elastic-stack).
Run the cells in order: they write six audit events to a file, start the EDOT Collector in
Docker with a restricted API key, reconstruct one actor's timeline with ES|QL, read back a
365-day retention policy, rerun the query on a mounted searchable snapshot, and clean up.

Requires Elasticsearch and Kibana 9.5+ on Elastic Cloud Hosted (for the `found-snapshots`
repository and the Enterprise license), Docker, and a `.env` copied from `.env.example` with a
superuser API key. To create one, in Kibana **Dev Tools**:

```
POST /_security/api_key
{ "name": "audit-lab-notebook", "role_descriptors": {} }
```

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

In [ ]:
import json
import os
import secrets
import subprocess
import time
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv()

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"
COLLECTOR_VERSION = os.getenv("COLLECTOR_VERSION", "9.5.3")
REPOSITORY = os.getenv("SNAPSHOT_REPOSITORY", "found-snapshots")

STREAM = "logs-audit_demo.otel-lab"  # logs-<dataset>.otel-<namespace>
TEMPLATE = "logs-audit_demo.otel-lab"
COLLECTOR_KEY_NAME = "audit-lab-collector"
SNAPSHOT = (
    "audit-article-snapshot-20260907"  # snapshot and mounted index share this name
)
ROLE = "audit_investigator"

es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)
KBN_HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}


def esql(query, title=None):
    """Run an ES|QL query and print the result as a text table."""
    response = es.esql.query(query=query, format="txt")
    if title:
        print(f"\n{title}")
    print(response.body)
    return response.body


info = es.info()["version"]["number"]
license_type = es.license.get()["license"]["type"]
print(f"elasticsearch {info}, license {license_type}")
if license_type not in ("enterprise", "trial"):
    print(
        "Searchable snapshots need an Enterprise (or trial) license; step 8 will be skipped."
    )

## 2. Write the audit events

Four events for `alex` (two failed logins, a successful login, a failed role change) and two for
`priya`. Timestamps are fixed on 2026-09-07 so the article's time window always matches.

In [ ]:
EVENTS = [
    {
        "timestamp": "2026-09-07T05:00:00Z",
        "event.id": "evt-0001",
        "user.name": "alex",
        "event.action": "login",
        "event.outcome": "failure",
        "audit.target": "admin-console",
        "source.address": "192.0.2.10",
    },
    {
        "timestamp": "2026-09-07T05:00:30Z",
        "event.id": "evt-0002",
        "user.name": "priya",
        "event.action": "login",
        "event.outcome": "success",
        "audit.target": "admin-console",
        "source.address": "192.0.2.25",
    },
    {
        "timestamp": "2026-09-07T05:01:00Z",
        "event.id": "evt-0003",
        "user.name": "alex",
        "event.action": "login",
        "event.outcome": "failure",
        "audit.target": "admin-console",
        "source.address": "192.0.2.10",
    },
    {
        "timestamp": "2026-09-07T05:02:00Z",
        "event.id": "evt-0004",
        "user.name": "alex",
        "event.action": "login",
        "event.outcome": "success",
        "audit.target": "admin-console",
        "source.address": "192.0.2.10",
    },
    {
        "timestamp": "2026-09-07T05:03:00Z",
        "event.id": "evt-0005",
        "user.name": "alex",
        "event.action": "role-change",
        "event.outcome": "failure",
        "audit.target": "billing-admin",
        "source.address": "192.0.2.10",
    },
    {
        "timestamp": "2026-09-07T05:04:30Z",
        "event.id": "evt-0006",
        "user.name": "priya",
        "event.action": "export",
        "event.outcome": "success",
        "audit.target": "billing-reports",
        "source.address": "192.0.2.25",
    },
]

os.makedirs("data", exist_ok=True)
FIXTURE_LINES = [json.dumps(event, separators=(",", ":")) for event in EVENTS]
with open("data/audit-events.ndjson", "w") as handle:
    handle.write("\n".join(FIXTURE_LINES) + "\n")

print(f"{len(FIXTURE_LINES)} events written to data/audit-events.ndjson")
print(FIXTURE_LINES[0])

## 3. Give the audit stream its own retention

A template for `logs-audit_demo.otel-*` that reuses the built-in `logs-otel@template`
components, prefers data stream lifecycle over ILM, and sets 365 days of retention.

In [ ]:
es.options(ignore_status=404).indices.delete(index=SNAPSHOT)
es.options(ignore_status=404).indices.delete_data_stream(name=STREAM)

base = es.indices.get_index_template(name="logs-otel@template")["index_templates"][0][
    "index_template"
]

es.indices.put_index_template(
    name=TEMPLATE,
    index_patterns=["logs-audit_demo.otel-*"],
    priority=500,
    data_stream={},
    composed_of=base.get("composed_of", []),
    ignore_missing_component_templates=base.get(
        "ignore_missing_component_templates", []
    ),
    template={
        "settings": {"index.lifecycle.prefer_ilm": False},
        "lifecycle": {"data_retention": "365d"},
    },
    meta={
        "description": "Audit logging lab: dedicated 365-day retention for the audit stream"
    },
)
print(
    f"template {TEMPLATE} composed of {len(base.get('composed_of', []))} component templates"
)

## 4. The collector's restricted API key

`create_doc` and `auto_configure` on the audit stream, `monitor` on the cluster, nothing else.
Set `COLLECTOR_API_KEY` in `.env` to use a key you created in Dev Tools:

```
POST /_security/api_key
{
  "name": "audit-lab-collector",
  "role_descriptors": {
    "audit_collector": {
      "cluster": ["monitor"],
      "indices": [{ "names": ["logs-audit_demo.otel-lab"], "privileges": ["create_doc", "auto_configure"] }]
    }
  }
}
```

Without it, the cell creates a collector user and role and issues the key as that user, since
an API key cannot create another key with privileges of its own.

In [ ]:
COLLECTOR_ROLE = "audit_collector"
COLLECTOR_USER = "audit-lab-collector"
COLLECTOR_PRIVILEGES = {
    "cluster": ["monitor"],
    "indices": [{"names": [STREAM], "privileges": ["create_doc", "auto_configure"]}],
}

COLLECTOR_API_KEY = os.getenv("COLLECTOR_API_KEY")
CREATED_COLLECTOR = {}  # what this cell created, so the cleanup only removes that

if COLLECTOR_API_KEY:
    print("using COLLECTOR_API_KEY from .env")
else:
    es.security.put_role(
        name=COLLECTOR_ROLE,
        cluster=COLLECTOR_PRIVILEGES["cluster"] + ["manage_own_api_key"],
        indices=COLLECTOR_PRIVILEGES["indices"],
    )
    collector_password = secrets.token_urlsafe(24)
    es.security.put_user(
        username=COLLECTOR_USER, password=collector_password, roles=[COLLECTOR_ROLE]
    )
    for key in es.security.get_api_key(username=COLLECTOR_USER).get("api_keys", []):
        if not key.get("invalidated"):
            es.security.invalidate_api_key(ids=[key["id"]])

    # Created as the collector user, so the key holds that user's privileges and nothing more.
    # The role descriptor narrows it further: the key itself cannot manage API keys.
    es_collector_user = Elasticsearch(
        ES_URL, basic_auth=(COLLECTOR_USER, collector_password), request_timeout=60
    )
    collector_key = es_collector_user.security.create_api_key(
        name=COLLECTOR_KEY_NAME, role_descriptors={COLLECTOR_ROLE: COLLECTOR_PRIVILEGES}
    )
    COLLECTOR_API_KEY = collector_key["encoded"]
    CREATED_COLLECTOR = {
        "key_id": collector_key["id"],
        "user": COLLECTOR_USER,
        "role": COLLECTOR_ROLE,
    }
    print(f"created API key {collector_key['id']} as user {COLLECTOR_USER}")

es_collector = Elasticsearch(ES_URL, api_key=COLLECTOR_API_KEY, request_timeout=60)
check = es_collector.security.has_privileges(
    index=[
        {
            "names": [STREAM],
            "privileges": [
                "create_doc",
                "auto_configure",
                "index",
                "write",
                "delete",
                "manage",
            ],
        }
    ]
)
print(json.dumps(check["index"][STREAM], indent=2))

with open(".env.docker", "w") as handle:
    handle.write(
        f"ELASTICSEARCH_URL={ES_URL}\nCOLLECTOR_API_KEY={COLLECTOR_API_KEY}\n"
        f"COLLECTOR_VERSION={COLLECTOR_VERSION}\n"
    )
print("collector credentials written to .env.docker")

## 5. Start the EDOT Collector

`docker-compose.yml` runs `elastic-otel-collector` with `otel-config.yaml`: `filelog` receiver,
`json_parser` operator, `resource` processor for dataset and namespace, `elasticsearch` exporter
in `otel` mode, offsets and queue in `file_storage`. Waits until the six records are searchable.

In [ ]:
COMPOSE = ["docker", "compose", "--env-file", ".env.docker"]


def compose(*args):
    result = subprocess.run(COMPOSE + list(args), capture_output=True, text=True)
    print(result.stdout, result.stderr)
    result.check_returncode()


def wait_for_count(index, expected, attempts=36):
    for _ in range(attempts):
        if es.options(ignore_status=404).indices.exists(index=index):
            es.indices.refresh(index=index)
            count = es.count(index=index)["count"]
            if count >= expected:
                return count
        time.sleep(5)
    raise RuntimeError(
        f"{index} did not reach {expected} documents; check `docker logs audit-lab-collector`"
    )


compose("down", "-v")
compose("up", "-d")
print(f"{wait_for_count(STREAM, len(EVENTS))} documents in {STREAM}")

The raw line is kept in `body.text` and the parsed fields land under `attributes`.

In [ ]:
hits = es.search(index=STREAM, size=20, sort="@timestamp")["hits"]["hits"]
stored = sorted(hit["_source"]["body"]["text"] for hit in hits)
print(
    f"{sum(line in FIXTURE_LINES for line in stored)} of {len(stored)} stored body.text strings match the fixture lines"
)

sample = hits[0]["_source"]
print(
    json.dumps(
        {
            "@timestamp": sample["@timestamp"],
            "attributes": sample["attributes"],
            "data_stream": sample["data_stream"],
        },
        indent=2,
    )
)

Restart the collector: still six records and six distinct event IDs.

In [ ]:
compose("restart")
time.sleep(20)
es.indices.refresh(index=STREAM)
esql(
    f"""
FROM {STREAM}
| STATS records = COUNT(*), distinct_event_ids = COUNT_DISTINCT(attributes.event.id)
""",
    "After restart",
)

## 6. Reconstruct one actor's timeline

Same query as Discover, with the time window in the `WHERE` clause.

In [ ]:
WINDOW = '@timestamp >= "2026-09-07T04:59:00Z" AND @timestamp < "2026-09-07T05:06:00Z"'

ACTOR_QUERY = f"""
FROM {{index}}
| WHERE {WINDOW} AND attributes.user.name == "alex"
| SORT @timestamp
| KEEP @timestamp, attributes.event.action, attributes.event.outcome,
       attributes.audit.target, attributes.source.address
| LIMIT 100
"""

esql(ACTOR_QUERY.format(index=STREAM), "alex, from the live data stream")

## 7. Read back the retention

Which mechanism manages the backing index, and what retention it applies.

In [ ]:
stream = es.indices.get_data_stream(name=STREAM)["data_streams"][0]
BACKING_INDEX = stream["indices"][0]["index_name"]
print(f"backing index {BACKING_INDEX}")
print(
    f"managed by: {stream['indices'][0].get('managed_by')}, prefer_ilm: {stream['indices'][0].get('prefer_ilm')}"
)

lifecycle = es.indices.get_data_lifecycle(name=STREAM)["data_streams"][0]["lifecycle"]
print(json.dumps(lifecycle, indent=2))
assert lifecycle["effective_retention"] == "365d"

## 8. Mount a searchable snapshot and rerun the query

Snapshot the backing index, mount it with `full_copy`, run the same query on the mount.

In [ ]:
if license_type not in ("enterprise", "trial"):
    print("skipped: searchable snapshots need an Enterprise or trial license")
else:
    if REPOSITORY not in es.snapshot.get_repository():
        raise RuntimeError(
            f"repository {REPOSITORY} not found; set SNAPSHOT_REPOSITORY in .env"
        )

    es.options(ignore_status=404).indices.delete(index=SNAPSHOT)
    es.options(ignore_status=404).snapshot.delete(
        repository=REPOSITORY, snapshot=SNAPSHOT
    )

    result = es.snapshot.create(
        repository=REPOSITORY,
        snapshot=SNAPSHOT,
        wait_for_completion=True,
        indices=BACKING_INDEX,
        include_global_state=False,
    )
    print(f"snapshot {SNAPSHOT}: {result['snapshot']['state']}")

    es.searchable_snapshots.mount(
        repository=REPOSITORY,
        snapshot=SNAPSHOT,
        index=BACKING_INDEX,
        renamed_index=SNAPSHOT,
        storage="full_copy",
        wait_for_completion=True,
        index_settings={"index.number_of_replicas": 0},
    )
    es.cluster.health(index=SNAPSHOT, wait_for_status="yellow", timeout="120s")
    esql(
        ACTOR_QUERY.format(index=SNAPSHOT), "alex, from the mounted searchable snapshot"
    )

## 9. Create the investigator role

`read` and `view_index_metadata` on the stream and the mount, plus Discover in the Kibana space.

In [ ]:
role = {
    "elasticsearch": {
        "cluster": [],
        "indices": [
            {"names": [STREAM, SNAPSHOT], "privileges": ["read", "view_index_metadata"]}
        ],
    },
    "kibana": [{"base": [], "feature": {"discover_v2": ["read"]}, "spaces": [SPACE]}],
}
response = requests.put(
    f"{KBN_BASE}/api/security/role/{ROLE}", headers=KBN_HEADERS, json=role, timeout=60
)
if response.status_code == 400 and "discover_v2" in response.text:
    role["kibana"][0]["feature"] = {
        "discover": ["read"]
    }  # feature id on older 9.x releases
    response = requests.put(
        f"{KBN_BASE}/api/security/role/{ROLE}",
        headers=KBN_HEADERS,
        json=role,
        timeout=60,
    )
response.raise_for_status()

print(json.dumps(es.security.get_role(name=ROLE)[ROLE]["indices"], indent=2))

## 10. Clean up

Stops the collector, deletes the mounted index and then the snapshot, the data stream, the
template, the investigator role, and the collector key, user and role if step 4 created them.

In [ ]:
compose("down", "-v")
es.options(ignore_status=404).indices.delete(index=SNAPSHOT)
es.options(ignore_status=404).snapshot.delete(repository=REPOSITORY, snapshot=SNAPSHOT)
es.options(ignore_status=404).indices.delete_data_stream(name=STREAM)
es.options(ignore_status=404).indices.delete_index_template(name=TEMPLATE)
es.options(ignore_status=404).security.delete_role(name=ROLE)
if CREATED_COLLECTOR:
    es.security.invalidate_api_key(ids=[CREATED_COLLECTOR["key_id"]])
    es.options(ignore_status=404).security.delete_user(
        username=CREATED_COLLECTOR["user"]
    )
    es.options(ignore_status=404).security.delete_role(name=CREATED_COLLECTOR["role"])
print("cleaned up")